In [ ]:
from util.DataGen import *
from util.Plotting import *
from util.Processing import *
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy.linalg import circulant

# Wiener filter
# Test filtering out gaussian noise and quantization noise

N = 1e4
kernel_magnitude = 1
noise_scale = 1e-2
noise_stdev = kernel_magnitude * noise_scale

N = int(N)
_, pulse = nai_pulse(kernel_magnitude, N)
kernel_fft = np.abs(np.fft.fft(pulse, n=N, norm='backward'))

noise = np.random.normal(0, noise_stdev, int(N))
fig, axes = plt.subplots(2,1, figsize=(10, 8), dpi=400)

noise_fft = np.abs(np.fft.fft(noise))
start = noise_fft.size//2
bins = N

axes[0].plot(noise, label='Noise')
axes[0].plot(pulse, label='Kernel')
axes[0].legend()

# noise = np.mean(noise_fft) # this should not change by length of fft...
print('noise var', np.var(noise))
noise = np.var(noise)
print('noise', noise)

axes[1].plot([0, N], [noise, noise], label='WG Noise expected value')
axes[1].plot(kernel_fft[:kernel_fft.size//2], label='Kernel FFT')
axes[1].set_yscale('log')
axes[1].set_xlim([0, kernel_fft.size//2])
axes[1].legend()


In [ ]:
N = 1000
n_pulses = 25
noise_stdev = 1
bits = 12

_, pulse = nai_pulse(1, 101)
kernel = pulse
n = kernel.size

# np.random.seed(0)
indeces = np.random.randint(0, N-n-1, n_pulses)
indeces = np.sort(indeces)
volts = np.random.uniform(1, 3, n_pulses)
volts = 10 ** volts

energies_ts = np.zeros(N)
for i, index in enumerate(indeces):
    energies_ts[index] += volts[i]

trace_fft = td_convolve(energies_ts, kernel)
trace_fft_noise = trace_fft + np.random.normal(0, noise_stdev, N)
trace_fft_quantized = quantize(trace_fft, bits)
trace_fft_noise_quantized = quantize(trace_fft_noise, bits) + np.random.normal(0, noise_stdev, N)

fig, axes = plt.subplots(5, 1, figsize=(10, 8), dpi=400)
fig.tight_layout()
plot_photons(axes[0], indeces, volts, label_='{} Photons'.format(len(volts)), color='r', alpha=.25)
axes[0].set_xlim([0, N])
axes[0].set_ylim([0, 1000])
axes[0].legend(loc=1)
axes[0].set_ylabel('Volts')

wiener_deconv_w_noise = wiener_deconvolve(trace_fft_noise, kernel, noise_stdev)
axes[1].plot(wiener_deconv_w_noise, label='Noise stdev={}: Wiener Deconv'.format(noise_stdev))
axes[1].set_xlim([0, N])
axes[1].set_ylim([0, 1000])
axes[1].legend(loc=1)
axes[1].set_ylabel('Volts')

wiener_deconv_quantized = wiener_deconvolve(trace_fft_quantized, kernel, noise_stdev)
axes[2].plot(wiener_deconv_quantized, label='bits={}: Wiener Deconv'.format(bits))
axes[2].set_xlim([0, N])
axes[2].set_ylim([0, 1000])
axes[2].legend(loc=1)
axes[2].set_ylabel('Volts')

wiener_deconv_noise_quantized = wiener_deconvolve(trace_fft_noise_quantized, kernel, noise_stdev)
axes[3].plot(wiener_deconv_noise_quantized, label='Noise stdev={}, bits={}: Wiener Deconv'.format(noise_stdev, bits))
axes[3].set_xlim([0, N])
axes[3].set_ylim([0, 1000])
axes[3].legend(loc=1)
axes[3].set_ylabel('Volts')

wiener_deconv_noise_quantized = fft_deconvolve(trace_fft_noise, kernel)
axes[4].plot(wiener_deconv_noise_quantized, label='Noise stdev={}, bits={}: FFT Deconv'.format(noise_stdev, bits))
axes[4].set_xlim([0, N])
axes[4].set_ylim([0, 1000])
axes[4].legend(loc=1)
axes[4].set_ylabel('Volts')

# print(np.sum(trace_fft_noise), np.sum(wiener_deconv_w_noise))
# print(np.sum(trace_fft_quantized), np.sum(wiener_deconv_quantized))
# print(np.sum(trace_fft_noise_quantized), np.sum(wiener_deconv_noise_quantized))